In [ ]:
import os
import io
import time
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import matplotlib.pyplot as plt
import seaborn as sns

import ee
import geemap

In [ ]:
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

In [ ]:
# ---- Project paths (matches your repo layout) -------------------------------
PROJECT_ROOT = Path.home() / "Documents" / "Projects" / "Fire_emission_rivers"
DATA_DIR     = PROJECT_ROOT / "data" / "Godavari_River"
OUT_DIR      = PROJECT_ROOT / "outputs" / "Godavari_River"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Basin inputs -------------------------------------------------------------
BASIN_GPKG = DATA_DIR / "Godavari_Basin.gpkg"      # preferred
BASIN_SHP  = DATA_DIR / "Godavari_Shape.shp"        # fallback


In [ ]:

# Load basin boundary
def load_basin_boundary():
    if BASIN_GPKG.exists():
        gdf = gpd.read_file(BASIN_GPKG)
    elif BASIN_SHP.exists():
        gdf = gpd.read_file(BASIN_SHP)
    else:
        raise FileNotFoundError(f"No basin boundary found at {BASIN_GPKG} or {BASIN_SHP}")

    gdf = gdf.to_crs(epsg=4326)
    gdf["dissolve_key"] = "Godavari"
    basin_gdf = gdf.dissolve(by="dissolve_key").reset_index(drop=True)
    basin_gdf["basin_name"] = "Godavari"
    return basin_gdf

basin_gdf = load_basin_boundary()
basin_bbox = tuple(basin_gdf.total_bounds)  # (minx, miny, maxx, maxy)
print("Basin bbox:", basin_bbox)
basin_gdf.plot(edgecolor="black", facecolor="tan", alpha=0.5, figsize=(6, 6))
plt.title("Godavari Basin Boundary")
plt.show()

### FIRMS Fire Activity Dataset

In [ ]:
# ---- NASA FIRMS API Config ----------------------------------------------------
# Get a free MAP KEY at: https://firms.modaps.eosdis.nasa.gov/api/map_key/
from dotenv import load_dotenv

load_dotenv()  # reads .env from the current working directory (your project root)
FIRMS_MAP_KEY = os.environ["FIRMS_MAP_KEY"]

if not FIRMS_MAP_KEY:
    raise ValueError("FIRMS_MAP_KEY not found - check your .env file")

START_YEAR = 2019
END_YEAR   = 2024

# Archive ("Standard Processing") sources - combine MODIS + VIIRS for full FRP coverage
FIRMS_SOURCES = [
    "MODIS_SP",          # Covers 2019–2024 fully (1km resolution)
    "VIIRS_SNPP_SP",     # Covers 2019–2024 fully (375m resolution)
    "VIIRS_NOAA20_SP"    # Covers 2019–2024 fully (375m resolution)
]

In [ ]:
'''import requests

# 1. Insert your actual key from https://firms.modaps.eosdis.nasa.gov/api/map_key/
TEST_KEY = os.environ["FIRMS_MAP_KEY"]  

# Test 5 days in 2023 for Godavari bbox
test_url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{TEST_KEY}/VIIRS_SNPP_SP/77.0,16.0,82.0,20.0/5/2023-04-01"

resp = requests.get(test_url)
print("Status Code:", resp.status_code)
print("Response Header/First 300 chars:")
print(resp.text[:300])'''

In [ ]:
# -------------------------------------------------------------------
# 1. fetch_firms_year(source, year, bbox, map_key)
# -------------------------------------------------------------------
# - Breaks 1 calendar year into 10-day start dates to respect FIRMS API single-query limits.
# - Sends HTTP GET requests for each 10-day chunk and parses valid CSV responses into DataFrames.
# - Combines the chunks into a single year-long DataFrame and tags it with the data source name.

def fetch_firms_year(source, year, bbox, map_key):
    """Fetch one calendar year of FIRMS archive detections by chunking in 10-day steps."""
    min_lon, min_lat, max_lon, max_lat = bbox
    area_str = f"{min_lon},{min_lat},{max_lon},{max_lat}"
    
    # 1. Break the entire year into 10-day starting dates
    start_dates = pd.date_range(start=f"{year}-01-01", end=f"{year}-12-31", freq="5D")
    year_dfs = []

    # 2. Loop through each 10-day chunk
    for s_date in start_dates:
        days_remaining = (pd.Timestamp(f"{year}-12-31") - s_date).days + 1
        day_range = min(5, days_remaining)  # Strictly kept under FIRMS 5-day API limit
        date_str = s_date.strftime("%Y-%m-%d")

        url = (
            f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
            f"{map_key}/{source}/{area_str}/{day_range}/{date_str}"
        )

        try:
            resp = requests.get(url, timeout=60)
            if resp.status_code == 200 and not resp.text.strip().lower().startswith("invalid"):
                chunk_df = pd.read_csv(io.StringIO(resp.text))
                if not chunk_df.empty:
                    year_dfs.append(chunk_df)
        except Exception as e:
            print(f"  [warn] Error fetching {source} for {date_str}: {e}")

        time.sleep(0.1)  # Short pause to protect rate limits

    if year_dfs:
        df = pd.concat(year_dfs, ignore_index=True).drop_duplicates()
        df["source"] = source
        return df

    return pd.DataFrame()

In [ ]:
# -------------------------------------------------------------------
# 2. fetch_firms_all_years(sources, start_year, end_year, bbox, map_key)
# -------------------------------------------------------------------
# - Runs a nested loop over all specified satellite sources and all target archive years.
# - Calls fetch_firms_year() for each combination, applying a short delay to respect rate limits.
# - Stacks every yearly DataFrame vertically into one consolidated master DataFrame for the entire period.

def fetch_firms_all_years(sources, start_year, end_year, bbox, map_key, pause=1.0):
    frames = []
    for source in sources:
        for year in range(start_year, end_year + 1):
            print(f"Fetching {source} {year} ...")
            df = fetch_firms_year(source, year, bbox, map_key)
            if not df.empty:
                frames.append(df)
            time.sleep(pause)  # be polite to the API
    if not frames:
        raise RuntimeError("No FIRMS data retrieved — check MAP_KEY / bbox / sources.")
    return pd.concat(frames, ignore_index=True)

In [ ]:
# -------------------------------------------------------------------
# 3. standardize_firms(df)
# -------------------------------------------------------------------
# - Zero-pads satellite acquisition time strings and builds a unified, parsed acq_datetime column.
# - Filters down to required attributes, standardizes coordinate column names (lat/lon), and casts numeric FRP.
# - Drops rows with missing coordinates or datetimes to produce a clean Pandas dataset ready for GeoPandas conversion.

def standardize_firms(df):
    """Keep the attributes needed by the workflow and build a proper datetime."""
    df = df.copy()
    df["acq_time"] = df["acq_time"].astype(str).str.zfill(4)
    df["acq_datetime"] = pd.to_datetime(
        df["acq_date"] + " " + df["acq_time"].str[:2] + ":" + df["acq_time"].str[2:],
        errors="coerce",
    )
    keep_cols = ["latitude", "longitude", "acq_date", "acq_time", "acq_datetime",
                 "frp", "confidence", "satellite", "instrument", "source"]
    keep_cols = [c for c in keep_cols if c in df.columns]
    df = df[keep_cols].rename(columns={"latitude": "lat", "longitude": "lon"})
    df["frp"] = pd.to_numeric(df["frp"], errors="coerce")
    df = df.dropna(subset=["lat", "lon", "acq_datetime", "frp"])
    return df


In [ ]:

firms_raw = fetch_firms_all_years(
    sources=FIRMS_SOURCES,
    start_year=START_YEAR,
    end_year=END_YEAR,
    bbox=basin_bbox,
    map_key=FIRMS_MAP_KEY,
)
print("Raw FIRMS records (bbox only):", firms_raw.shape)

In [ ]:
firms_std = standardize_firms(firms_raw)
firms_gdf = gpd.GeoDataFrame(
    firms_std,
    geometry=gpd.points_from_xy(firms_std["lon"], firms_std["lat"]),
    crs="EPSG:4326",
)

# Precise spatial clip to the actual basin polygon (bbox above was just coarse pre-filter)
firms_basin = gpd.sjoin(firms_gdf, basin_gdf[["basin_name", "geometry"]], predicate="within", how="inner")
firms_basin = firms_basin.drop(columns=["index_right"])

print(f"FIRMS detections in bbox: {len(firms_gdf)}  ->  within Godavari basin: {len(firms_basin)}")
firms_basin.head()

In [ ]:

# Save Stage output
firms_out_path = OUT_DIR / f"firms_godavari_{START_YEAR}_{END_YEAR}.gpkg"
firms_basin.to_file(firms_out_path, driver="GPKG")
firms_basin.drop(columns="geometry").to_csv(
    OUT_DIR / f"firms_godavarGodi_{START_YEAR}_{END_YEAR}.csv", index=False
)
print("Saved:", firms_out_path)

### Earth Engine

In [ ]:
EE_PROJECT = "fluid-blade-497705-q1"

In [ ]:
try:
    ee.Initialize(project=EE_PROJECT)
    print("EE already authenticated — initialized successfully.")
except Exception:
    print("No valid credentials found — launching browser auth...")
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
    print("EE authenticated and initialized.")